# Diabetes model for TrustyAI SPD demo

This notebook rebuilds the diabetes model so that the raw `Age` feature becomes a binary `AgeOver50` feature.

## Why this version is better for TrustyAI SPD

- `AgeOver50` is a stable binary protected attribute with values `0` and `1`
- the model is trained on **raw input values**, so the serving payload can also use raw values
- the ONNX export keeps the same basic deployment shape:
  - input: `dense_input`
  - outputs: `label` and `probabilities`

This makes the model much easier to use as a fairness demo with TrustyAI.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## Configuration

Set `ENABLE_SYNTHETIC_BIAS = True` if you want a stronger TrustyAI SPD demo signal.

When enabled, the notebook intentionally flips a portion of favorable outcomes for the `AgeOver50 == 1` group in the training set. This is only for a lab/demo scenario.


In [ ]:
%pwd

In [ ]:
DATA_PATH = Path("diabetes-model/pima-indians-diabetes.data.csv")
MODEL_PATH = Path("diabetes.onnx")
TRAINING_DATA_EXPORT = Path("diabetes-trustyai-training-data.csv")
SAMPLE_PAYLOAD_DIR = Path("sample-payloads")

RANDOM_STATE = 42
TEST_SIZE = 0.20
EPOCHS = 200
LEARNING_RATE = 0.01

# Demo-only option
ENABLE_SYNTHETIC_BIAS = False
SYNTHETIC_BIAS_FLIP_RATE = 0.20


## Load dataset

Original columns:
- Pregnancies
- Glucose
- BloodPressure
- SkinThickness
- Insulin
- BMI
- DiabetesPedigreeFunction
- Age
- Outcome


In [ ]:
names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

df = pd.read_csv(DATA_PATH, names=names)
print(df.head())
print()
print(df.describe(include="all"))


## Replace `Age` with `AgeOver50`

This keeps the model at 8 input features while changing the protected attribute into a binary feature that works much better with SPD.


In [ ]:
df["AgeOver50"] = (df["Age"] > 50).astype(np.float32)
df = df.drop(columns=["Age"])

feature_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "AgeOver50",
]

target_col = "Outcome"

X_df = df[feature_cols].copy()
y = df[target_col].astype(np.int64).values

print("Feature columns:", feature_cols)
print()
print("AgeOver50 value counts:")
print(df["AgeOver50"].value_counts().sort_index())


## Train/test split

This version deliberately does **not** standardize the features.

That keeps the deployment simple:
- training uses raw values
- serving uses raw values
- TrustyAI sees the same raw `AgeOver50` values that the model uses


In [ ]:
X = X_df.values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_df = pd.DataFrame(X_train, columns=feature_cols)
test_df = pd.DataFrame(X_test, columns=feature_cols)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


## Optional demo-only synthetic bias

If enabled, this creates a clearer SPD signal by reducing favorable outcomes for the `AgeOver50 == 1` group.

By default, this notebook treats `Outcome == 0` ("no diabetes") as the favorable outcome for fairness demo purposes.


In [ ]:
y_train_effective = y_train.copy()

if ENABLE_SYNTHETIC_BIAS:
    older_mask = train_df["AgeOver50"].values == 1.0
    favorable_mask = y_train_effective == 0

    candidate_idx = np.where(older_mask & favorable_mask)[0]
    rng = np.random.default_rng(RANDOM_STATE)
    flip_count = int(len(candidate_idx) * SYNTHETIC_BIAS_FLIP_RATE)

    if flip_count > 0:
        flip_idx = rng.choice(candidate_idx, size=flip_count, replace=False)
        y_train_effective[flip_idx] = 1

    print(f"Synthetic bias enabled. Flipped {flip_count} labels from 0 -> 1 for AgeOver50 == 1.")
else:
    print("Synthetic bias disabled.")


## Define the model


In [ ]:
class DiabetesModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):
        logits = self.net(x)
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(probs, dim=1)
        return labels, probs


model = DiabetesModel()
model


## Train the model


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train_effective)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    loss = criterion(model.net(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:03d}/{EPOCHS} - loss: {loss.item():.4f}")


## Evaluate


In [ ]:
model.eval()

with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test)
    pred_labels, pred_probs = model(X_test_t)

y_pred = pred_labels.numpy()
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(float(accuracy), 4))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))


## Export ONNX

The export keeps the same inferencing pattern you already use:
- input name: `dense_input`
- output names: `label`, `probabilities`


In [ ]:
dummy_input = torch.randn(1, len(feature_cols), dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    MODEL_PATH.as_posix(),
    verbose=False,
    input_names=["dense_input"],
    output_names=["label", "probabilities"],
    dynamic_axes={
        "dense_input": {0: "batch_size"},
        "label": {0: "batch_size"},
        "probabilities": {0: "batch_size"},
    },
    opset_version=11,
)

print(f"Saved ONNX model to: {MODEL_PATH.resolve()}")


## Export training data for TrustyAI

This file includes the final feature names the deployed model expects, including `AgeOver50`.


In [ ]:
export_df = pd.DataFrame(X, columns=feature_cols)
export_df["Outcome"] = y

export_df.to_csv(TRAINING_DATA_EXPORT, index=False)
print(f"Saved training data export to: {TRAINING_DATA_EXPORT.resolve()}")
print(export_df.head())


## Create sample inference payloads

These payloads use the deployed v2 inference format and raw values.

They are useful for quick manual testing and for generating observations that TrustyAI can score later.


In [ ]:
SAMPLE_PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

sample_under_50 = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [1, 8],
            "datatype": "FP32",
            "data": [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0],
        }
    ],
}

sample_over_50 = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [1, 8],
            "datatype": "FP32",
            "data": [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 1.0],
        }
    ],
}

sample_batch = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [4, 8],
            "datatype": "FP32",
            "data": [
                1.0, 95.0, 70.0, 20.0, 85.0, 24.0, 0.20, 0.0,
                3.0, 165.0, 88.0, 30.0, 220.0, 37.5, 0.65, 0.0,
                6.0, 180.0, 92.0, 35.0, 260.0, 41.2, 0.80, 1.0,
                1.0, 105.0, 68.0, 18.0, 95.0, 26.0, 0.25, 1.0,
            ],
        }
    ],
}

for name, payload in {
    "under50.json": sample_under_50,
    "over50.json": sample_over_50,
    "batch.json": sample_batch,
}.items():
    path = SAMPLE_PAYLOAD_DIR / name
    path.write_text(json.dumps(payload, indent=2))
    print(f"Saved {path.resolve()}")
